In [2]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import pandas as pd
import muon as mu
import scanpy as sc
import scirpy as ir
np.random.seed(42)
import random
random.seed(42)

import sys
sys.path.append("/ihome/ylee/yiz133/Code/Data processing/functions")
import importlib
import mdata_utils

In [3]:
importlib.reload(mdata_utils)

<module 'mdata_utils' from '/ihome/ylee/yiz133/Code/Data processing/functions/mdata_utils.py'>

# Merge the data

In [4]:
%cd "/ix1/ylee/Yifan_Zhang/Code_data/external"

/ix1/ylee/Yifan_Zhang/Code_data/external


In [5]:
from pathlib import Path
import re

# Get all folders that don't start with '_'
current_dir = Path('.')
folders = [f for f in current_dir.iterdir() if f.is_dir() and not f.name.startswith('_')]

# Load yz_processed.h5mu from each folder into named variables
mdata_dict = {}
mdata_list = []

# select data sets as test 
test_folders = ['GSE188320 Th17']
n_top_genes = 200

for folder in folders:
    h5mu_file = folder / 'yz_processed_allGenes_annotateByScore.h5mu'
    
    if h5mu_file.exists():
        print(f"Loading: {h5mu_file}")
        mdata = mu.read(h5mu_file)
        var_name = re.sub(r'[^\w]', '_', folder.name) + '_mdata'
        
        # # reduce number of vars
        # sc.pp.highly_variable_genes(mdata['gex'], n_top_genes=n_top_genes)
        # mdata.mod['gex'] = mdata['gex'][:, mdata['gex'].var['highly_variable']].copy()
        # mdata = mdata_utils.sync_mdata_obs(mdata)

        # assign 'test' or 'train'
        if folder.name in test_folders:
            mdata.obs['set'] = 'test'
        else:
            mdata.obs['set'] = 'train'
            
        # Store in dictionary and list
        mdata_dict[var_name] = mdata
        globals()[var_name] = mdata  # Create variable in global namespace
        mdata_list.append(mdata)

    else:
        print(f"Skipping {folder.name}: yz_processed.h5mu not found")

Loading: GSE1 LEE/yz_processed_allGenes_annotateByScore.h5mu
Loading: GSE156718 CD4 iLN/yz_processed_allGenes_annotateByScore.h5mu
Loading: GSE293883 CD4 Tr mem day60/yz_processed_allGenes_annotateByScore.h5mu
Loading: GSE188320 Th17/yz_processed_allGenes_annotateByScore.h5mu


In [6]:
import mudata as md

merged = mdata_utils.merge_mdatas(mdata_list, mods=["gex","airr"], keep_obs_columns="common", index_unique=None)
merged

MuData object with n_obs × n_vars = 105261 × 16062
  obs:	'GSE', 'GSM', 'VDJ_1_cdr3_aa', 'VDJ_1_j_call', 'VDJ_1_v_call', 'VJ_1_cdr3_aa', 'VJ_1_j_call', 'VJ_1_v_call', 'cell_type', 'condition', 'sample_id', 'set', 'state'
  2 modalities
    airr:	105261 x 0
      obs:	'sample_id', 'receptor_type', 'receptor_subtype', 'chain_pairing', 'clone_id', 'clone_id_size', 'clonal_expansion'
      obsm:	'airr', 'chain_indices'
    gex:	105261 x 16062
      obs:	'sample_id', 'date', 'tissue', 'sample', 'mouse_BC', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_genes', 'n_counts', 'CD4score', 'CD8score', 'Tregscore', 'Th17score', 'cell_type', 'IFN_stimscore', 'Activationscore', 'Exhaustscore', 'Mem_Naivescore', 'state', 'TCR_clonotype_frequency', 'batch', 'condition'

In [7]:
# reduce number of vars
sc.pp.highly_variable_genes(merged['gex'], n_top_genes=5000)
merged.mod['gex'] = merged['gex'][:, merged['gex'].var['highly_variable']].copy()
merged = mdata_utils.sync_mdata_obs(merged)
        
GEX_df = merged['gex'].to_df()
# GEX_df

In [8]:
merged.obs['set'].value_counts()

# booll = merged.obs['GSE'] == 'GSE156718'
# m2 = merged[booll.to_numpy(dtype=bool)]

set
train    65503
test     39758
Name: count, dtype: Int64

In [9]:
for m in mdata_list:
    intesect_num = m.var_names.intersection(merged.var_names)
    print(intesect_num)

Index(['4732440D04Rik', 'Rb1cc1', 'Pcmtd1', 'Sgk3', 'Cspp1', 'Arfgef1',
       'Ncoa2', 'Rdh10', 'Il17a', 'Il17f',
       ...
       'Uty', 'Ddx3y', 'Gm47283', 'mt-Atp8', 'mt-Atp6', 'mt-Co3', 'mt-Nd3',
       'mt-Nd5', 'mt-Nd6', 'mt-Cytb'],
      dtype='object', length=4548)
Index(['Rb1cc1', '4732440D04Rik', 'Pcmtd1', 'Sgk3', 'Cspp1', 'Arfgef1',
       'Ncoa2', 'Rdh10', 'Il17a', 'Il17f',
       ...
       'Zfp950', 'mt-Atp8', 'mt-Atp6', 'mt-Co3', 'mt-Nd3', 'mt-Nd5', 'mt-Nd6',
       'mt-Cytb', 'Tmlhe', 'AC149090.1'],
      dtype='object', length=3202)
Index(['4732440D04Rik', 'Rb1cc1', 'Pcmtd1', 'Sgk3', 'Cspp1', 'Arfgef1',
       'Ncoa2', 'Rdh10', 'Il17a', 'Il17f',
       ...
       'Uty', 'Ddx3y', 'Gm47283', 'mt-Atp8', 'mt-Atp6', 'mt-Co3', 'mt-Nd3',
       'mt-Nd5', 'mt-Nd6', 'mt-Cytb'],
      dtype='object', length=3945)
Index(['Rb1cc1', '4732440D04Rik', 'Pcmtd1', 'Sgk3', 'Cspp1', 'Arfgef1',
       'Ncoa2', 'Rdh10', 'Il17a', 'Il17f',
       ...
       'mt-Atp6', 'mt-Co3', 'mt-Nd3', 'm

# cluster by GEX

In [ ]:
sc.pp.pca(merged["gex"], n_comps=50)

### no batch correction

In [ ]:
sc.pp.neighbors(merged["gex"], n_neighbors = 50)
sc.tl.umap(merged["gex"], min_dist=0.5, spread=1.0)
sc.tl.leiden(merged["gex"], resolution = 1, n_iterations=-1, flavor = 'igraph')

In [ ]:
merged["gex"].obs['GSE'] = merged.obs['GSE']

In [ ]:
sc.pl.umap(merged["gex"], color=['state', 'tissue', 'cell_type', 'GSE'], ncols=2,)

### harmony correction

In [ ]:
X = np.asarray(merged["gex"].obsm['X_pca'], dtype=np.float64)
bad = ~np.isfinite(X).all(axis=1)
if bad.any():
    merged["gex"] = merged["gex"][~bad].copy()
    X = np.asarray(merged["gex"].obsm['X_pca'], dtype=np.float64)
merged["gex"].obsm['X_pca'] = np.ascontiguousarray(X)

# merged["gex"].obsm['X_pca'] = np.asarray(merged["gex"].obsm['X_pca'], dtype=np.float64)
# merged["gex"].obs['GSE'] = merged["gex"].obs['GSE'].astype("string").fillna("unknown").astype("category")
merged["gex"].obs['GSE'] = merged["gex"].obs['GSE'].astype("object").astype("category")

In [ ]:
## Batch correction
import scanpy.external as sce
sce.pp.harmony_integrate(merged["gex"], key="GSE",  basis='X_pca')

In [ ]:
merged["gex"]

In [ ]:
# sc.pp.neighbors(merged["gex"], n_neighbors = 50, use_rep='X_pca_harmony')
# sc.tl.umap(merged["gex"], min_dist=0.5, spread=1.0)
sc.pl.umap(merged["gex"], color=['tissue', 'GSE'], ncols=2,)

In [ ]:
merged['gex'].obs["GSE"] = merged['gex'].obs["GSE"].astype(str).astype("category")
merged.write('merged_EAE.h5mu')